# 03｜從商業問題到洞察（對應課程 M5）

**本 Notebook 的目標**：用 pandas 與 scipy 完成 M5 的三個題目與彩蛋題。

| 題目 | 方法 |
| --- | --- |
| A 顧客分群 | RFM 計分與分群規則 |
| B 配送與評分 | 分組比較、Welch t 檢定、效果大小、分層比較、迴歸 |
| C 賣家分級 | 延遲率排名與「如果改善」的試算 |
| 彩蛋 | 評論文字的關鍵字分類＋抽樣人工驗證 |

> **先有假設，再跑數字**。下面每一題都先寫下要驗證的假設，再寫程式。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import islemart_lib as L

L.setup_chinese_font()
pd.set_option("display.width", 160)
BLUE, ORANGE, GRAY = "#2E7D9A", "#E07A3F", "#9AA5AD"

T = L.load_all()
wide = L.build_orders_wide(T)        # 和 Notebook 02 同一套清理邏輯
delivered = wide[wide["order_status"] == "delivered"]
print(f"訂單 {len(wide):,} 筆，已送達 {len(delivered):,} 筆")

## 題目 A｜RFM 顧客分群

**假設**：少數顧客貢獻大部分營收；不同群的顧客需要不同的行銷方式。

**定義**
- 單位：同一個人（`customer_unique_id`），不是 `customer_id`
- 基準日：2026-01-01（資料最後一天的隔天）
- 只算已送達訂單；金額用本課的營收定義 `item_total`

In [ ]:
BASE_DATE = pd.Timestamp("2026-01-01")

rfm = delivered.groupby("customer_unique_id").agg(
    last_purchase=("order_purchase_timestamp", "max"),
    F=("order_id", "count"),
    M=("item_total", "sum"),
)
rfm["R"] = (BASE_DATE - rfm["last_purchase"]).dt.days
print(f"顧客數：{len(rfm):,}")
rfm["F"].value_counts().sort_index().rename("人數").to_frame().T

**計分方式**
- R：最近一次購買距今天數，分 5 等分，**越近分數越高**
- F：多數人只買過 1 次，等分會失效，所以直接用次數（1–5 次，5 次以上算 5 分）
- M：消費金額分 5 等分，越高分數越高

`rank(method="first")` 先把同分的人排出順序，`qcut` 才能切得平均。

In [ ]:
rfm["R_score"] = pd.qcut(rfm["R"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_score"] = rfm["F"].clip(upper=5)
rfm["M_score"] = pd.qcut(rfm["M"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment(r):
    if r.R_score >= 4 and r.F_score >= 2 and r.M_score >= 4:
        return "冠軍顧客"
    if r.F_score >= 2:
        return "忠誠顧客"
    if r.R_score >= 4:
        return "新顧客"
    if r.R_score <= 2 and r.M_score >= 4:
        return "沉睡高價值"
    if r.R_score <= 2:
        return "流失顧客"
    return "一般顧客"

rfm["分群"] = rfm.apply(segment, axis=1)

ACTIONS = {"冠軍顧客": "會員專屬禮遇、新品搶先購", "忠誠顧客": "點數加倍，推升客單價",
           "新顧客": "第二單優惠，促成回購", "沉睡高價值": "個人化喚回優惠",
           "流失顧客": "低成本觸及（電子報）", "一般顧客": "檔期活動通知"}

seg = rfm.groupby("分群").agg(人數=("M", "size"), 平均消費=("M", "mean"), 營收=("M", "sum"))
seg["人數占比%"] = (seg["人數"] / seg["人數"].sum() * 100).round(1)
seg["營收占比%"] = (seg["營收"] / seg["營收"].sum() * 100).round(1)
seg["平均消費"] = seg["平均消費"].round(0)
seg["行銷建議"] = seg.index.map(ACTIONS)
seg.sort_values("營收", ascending=False).drop(columns="營收")

In [ ]:
s = seg.sort_values("營收占比%")
fig, ax = plt.subplots(figsize=(9, 4.5))
y = np.arange(len(s))
ax.barh(y - 0.2, s["人數占比%"], height=0.38, color=GRAY, label="人數占比")
ax.barh(y + 0.2, s["營收占比%"], height=0.38, color=BLUE, label="營收占比")
ax.set_yticks(y, s.index)
ax.set_xlabel("%")
ax.set_title("各顧客群的人數與營收占比", loc="left")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

**解讀方式**：營收占比遠高於人數占比的群，就是要優先照顧的顧客。
**限制**：分群規則是人訂的，門檻改了，群的大小就會變；報告裡要寫清楚規則。

## 題目 B｜配送與評分

**假設 H1**：延遲到貨會讓評分變低，而且延遲越久，分數越低。

只看「有效送達且有評分」的訂單。

In [ ]:
d = wide[wide["is_valid_delivery"] & wide["review_score"].notna()].copy()
d["is_late"] = d["is_late"].astype(bool)

summary = d.groupby("is_late")["review_score"].agg(["count", "mean", "std"]).round(3)
summary.index = summary.index.map({False: "準時", True: "延遲"})
summary

### B-1 統計檢定與效果大小

- **Welch t 檢定**：兩組平均數的差異是否可能只是巧合（不假設兩組變異數相同）
- **Mann-Whitney U 檢定**：評分是 1–5 的等級資料，用不看平均數的方法再確認一次
- **Cohen's d**：差異有多大。p 值只告訴你「是不是巧合」，d 才告訴你「重不重要」（0.2 小、0.5 中、0.8 大）

In [ ]:
on_time = d.loc[~d["is_late"], "review_score"]
late = d.loc[d["is_late"], "review_score"]

t = stats.ttest_ind(on_time, late, equal_var=False)
u = stats.mannwhitneyu(on_time, late, alternative="two-sided")
pooled_sd = np.sqrt(((len(on_time) - 1) * on_time.var() + (len(late) - 1) * late.var())
                    / (len(on_time) + len(late) - 2))
cohen_d = (on_time.mean() - late.mean()) / pooled_sd

print(f"平均分差：{on_time.mean() - late.mean():.2f} 分")
print(f"Welch t = {t.statistic:.1f}，p = {t.pvalue:.2e}")
print(f"Mann-Whitney p = {u.pvalue:.2e}")
print(f"Cohen's d = {cohen_d:.2f}")

### B-2 劑量反應：延遲越久，分數越低嗎？

In [ ]:
bins = [-np.inf, 0, 2, 4, np.inf]
labels = ["準時", "延遲 1–2 天", "延遲 3–4 天", "延遲 5 天以上"]
d["延遲分組"] = pd.cut(d["late_days"], bins=bins, labels=labels)
dose = d.groupby("延遲分組", observed=True)["review_score"].agg(["count", "mean"]).round(2)
dose["低分比例%"] = d.groupby("延遲分組", observed=True)["review_score"].apply(lambda s: (s <= 2).mean() * 100).round(1)
dose

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = [BLUE] + [ORANGE] * 3
ax.bar(dose.index.astype(str), dose["mean"], color=colors, width=0.55)
for x, (m, n) in enumerate(zip(dose["mean"], dose["count"])):
    ax.text(x, m + 0.05, f"{m:.2f}\n(n={n:,})", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, 5.2)
ax.set_ylabel("平均評分")
ax.set_title("延遲越久，平均評分越低", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

### B-3「慢」不等於「延遲」：縣市層級看兩個指標

- **平均配送天數**：營運速度
- **延遲率**：承諾有沒有兌現

哪一個和評分的關係比較強？用 Spearman 等級相關係數比較。

In [ ]:
county = d.groupby("customer_county").agg(
    訂單數=("order_id", "count"),
    平均配送天數=("delivery_days", "mean"),
    延遲率=("is_late", "mean"),
    平均評分=("review_score", "mean"),
)
county["延遲率"] = county["延遲率"] * 100
county = county.round(2).sort_values("延遲率", ascending=False)

r_days = stats.spearmanr(county["平均配送天數"], county["平均評分"])
r_late = stats.spearmanr(county["延遲率"], county["平均評分"])
print(f"評分 vs 平均配送天數：rho = {r_days.statistic:.2f}")
print(f"評分 vs 延遲率：　　　rho = {r_late.statistic:.2f}")
county.head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, col, rho, unit in [(axes[0], "平均配送天數", r_days.statistic, "天"),
                           (axes[1], "延遲率", r_late.statistic, "%")]:
    ax.scatter(county[col], county["平均評分"], s=county["訂單數"] / 40, color=BLUE, alpha=0.75,
               edgecolor="white", linewidth=1)
    for name, row in county.iterrows():
        if name in ("金門縣", "台北市", "高雄市"):
            ax.annotate(name, (row[col], row["平均評分"]), xytext=(5, 4), textcoords="offset points", fontsize=9)
    ax.set_xlabel(f"{col}（{unit}）")
    ax.set_title(f"評分 vs {col}（rho = {rho:.2f}）", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("平均評分")
plt.tight_layout(); plt.show()

**解讀**：離島配送最慢，評分卻不差，因為承諾天數較長、多半準時；評分跟著「延遲率」走，而不是「天數」。
管理上的意義：改善評分不一定要更快，**承諾要設得準**。

### B-4 相關不等於因果：分層比較與迴歸

檔期爆量、商品類別可能同時影響延遲與評分（干擾因素）。檢查方法：
1. **分層比較**：在「同一類別 × 同一月份」裡面比較準時與延遲，再加權平均
2. **迴歸**：同時放入延遲、類別、月份，看延遲的係數還剩多少

In [ ]:
diffs = []
for (cat, month), g in d.groupby(["main_category", "purchase_month"]):
    a, b = g.loc[~g["is_late"], "review_score"], g.loc[g["is_late"], "review_score"]
    if len(a) >= 10 and len(b) >= 10:
        diffs.append({"n": len(g), "diff": a.mean() - b.mean()})
diffs = pd.DataFrame(diffs)
stratified = np.average(diffs["diff"], weights=diffs["n"])
print(f"整體平均分差：{on_time.mean() - late.mean():.2f}")
print(f"分層後加權平均分差：{stratified:.2f}（共 {len(diffs)} 個類別×月份分層）")

In [ ]:
# 迴歸：評分 = b0 + b1×是否延遲 + 類別虛擬變數 + 月份虛擬變數
X = pd.get_dummies(d[["main_category", "purchase_month"]], drop_first=True, dtype=float)
X.insert(0, "is_late", d["is_late"].astype(float))
X.insert(0, "const", 1.0)
coef, *_ = np.linalg.lstsq(X.to_numpy(), d["review_score"].to_numpy(dtype=float), rcond=None)
print(f"控制類別與月份後，延遲的係數：{coef[1]:.2f} 分")

**結論寫法**：「控制類別與月份後，延遲訂單的評分仍平均低約 X 分，效果大（d ≈ Y）。
這是觀察資料，最強的證據仍需實驗，例如調整承諾天數做 A/B 測試。」

## 題目 C｜賣家分級

**假設 H3**：少數賣家的延遲拖累整體評分。

用「主要賣家」（訂單第一個品項的賣家）代表該訂單的出貨賣家；只看有效送達訂單 50 筆以上的賣家。

In [ ]:
sellers = d.groupby("main_seller_id").agg(
    訂單數=("order_id", "count"),
    延遲率=("is_late", "mean"),
    平均評分=("review_score", "mean"),
)
sellers = sellers[sellers["訂單數"] >= 50]
median_late = sellers["延遲率"].median()
top10 = sellers.sort_values("延遲率", ascending=False).head(10).copy()

print(f"訂單 50 筆以上的賣家：{len(sellers)} 家，延遲率中位數 {median_late:.1%}")
top10.assign(延遲率=lambda x: (x["延遲率"] * 100).round(1), 平均評分=lambda x: x["平均評分"].round(2))

**如果這 10 家的延遲率降到中位數，整體評分會改善多少？**

試算邏輯：少掉的延遲訂單數 × 準時與延遲的分差 ÷ 全部有評分的訂單數。

In [ ]:
gap = on_time.mean() - late.mean()
fewer_late = ((top10["延遲率"] - median_late) * top10["訂單數"]).sum()
lift = fewer_late * gap / len(d)
print(f"減少延遲訂單約 {fewer_late:,.0f} 筆")
print(f"整體平均評分：{d['review_score'].mean():.3f} → 約 {d['review_score'].mean() + lift:.3f}（+{lift:.3f}）")

**解讀**：這是粗估，假設「改善延遲」會讓分數變成準時訂單的水準。
報告時要寫出這個假設，並建議先輔導 1–2 家賣家驗證效果。

## 彩蛋｜評論文字分類（關鍵字規則）

課堂上請 AI 做情緒分類；傳統做法是**關鍵字規則**。優點是透明、可重現；缺點是死板，遇到沒列到的說法就判斷不了。
一則評論可以同時屬於多個類別。

In [ ]:
RULES = {
    "物流延遲": ["等了好久", "物流", "延遲", "預計到貨日"],
    "商品瑕疵": ["破損", "刮傷", "瑕疵"],
    "與描述不符": ["不符", "落差", "退貨"],
    "缺件": ["只收到部分"],
    "客服": ["客服", "沒有回應"],
    "正面": ["滿意", "推薦", "很快", "CP值", "開心", "不錯", "早到", "回購"],
}

def classify(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return [k for k, words in RULES.items() if any(w in text for w in words)] or ["其他"]

rv = d[d["review_text"].notna()].copy()
rv["類別"] = rv["review_text"].map(classify)
exploded = rv.explode("類別").reset_index(drop=True)
print(f"有文字的評論：{len(rv):,} 則（{len(rv) / len(d):.1%}）")
pd.crosstab(exploded["類別"], exploded["review_score"], margins=True, margins_name="合計")

**抽樣人工驗證**：隨機抽 30 則，存成 CSV，由人逐筆標記分類是否正確，算出準確率。

In [ ]:
sample = rv.sample(30, random_state=42)[["order_id", "review_score", "review_text", "類別"]]
sample["類別"] = sample["類別"].map("、".join)
sample["人工判斷正確？(Y/N)"] = ""
sample.to_csv("review_sample_for_check.csv", index=False, encoding="utf-8-sig")
sample.head(8)

## 小結

| 課堂上的 AI 做法 | 本 Notebook 的傳統做法 |
| --- | --- |
| 請 AI 計算 RFM 並分群 | `groupby` 算 R/F/M，`qcut` 計分，規則函式分群 |
| 請 AI 做統計檢定 | `scipy.stats`：Welch t、Mann-Whitney、Spearman；自己算 Cohen's d |
| 追問「這是因果嗎」 | 分層比較＋ `numpy.linalg.lstsq` 迴歸 |
| AI 情緒分類 | 關鍵字規則＋抽樣人工驗證 |

**練習**：把 RFM 的基準日改成 2025-07-01，只用上半年的資料，分群結果有什麼不同？